# Day 8. Token / Vocabulary / Tokenizer

2주차 시작! 이번 주는 **NLP 기초 + Attention**입니다.

**오늘의 목표**
- 텍스트를 왜 숫자로 바꿔야 하는지 설명할 수 있다
- 글자 단위 Tokenizer를 직접 만들고 `encode` / `decode`를 구현한다 (커리큘럼 6-1과 같은 코드!)
- 글자 / 단어 / Subword 방식의 장단점을 비교할 수 있다
- BPE가 어떻게 Vocabulary를 만드는지 작은 예제로 직접 돌려본다
- 실제 GPT Tokenizer로 한국어가 왜 Token을 더 많이 쓰는지 확인한다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
from collections import Counter

## 1. 왜 텍스트를 숫자로 바꿔야 할까?

1주차에 배운 모든 연산은 **숫자**에만 동작합니다.
- Day 1: 데이터는 벡터(숫자 묶음)
- Day 2: 행렬 곱
- Day 3: `nn.Linear`, 그리고 마지막 `lm_head`가 내는 `vocab_size`개의 점수
- Day 5: 그 점수에 Softmax를 해서 "다음 Token" 확률

그런데 `"안녕하세요"`는 숫자가 아니죠. 그래서 맨 앞에 **텍스트 ↔ 숫자 변환기**가 필요합니다. 이게 **Tokenizer**입니다.

```text
"안녕하세요"  ──encode──▶  [5, 9, 12, 7, 10]  ──▶  모델  ──▶  [7]  ──decode──▶  "요"
```

용어 정리:
| 용어 | 뜻 |
|---|---|
| **Token** | 텍스트를 자른 조각 하나 (글자, 단어, 단어 조각 등) |
| **Vocabulary (Vocab)** | 모델이 아는 Token 전체 목록 |
| **Token ID** | Vocab 안에서 그 Token의 번호 |
| **encode** | 텍스트 → Token ID 리스트 |
| **decode** | Token ID 리스트 → 텍스트 |

Day 3의 `lm_head = nn.Linear(n_embd, vocab_size)`에서 `vocab_size`가 바로 **Vocab에 든 Token 개수**입니다.

## 2. 글자 단위(Character) Tokenizer 직접 만들기

가장 단순한 방법: **글자 하나 = Token 하나**.
이 코드는 커리큘럼 **6-1 Character Tokenizer**와 똑같습니다. 4주차 mini GPT에서 그대로 다시 씁니다.

In [ ]:
text = "안녕하세요. 오늘 날씨가 좋네요. Hello world!"

chars = sorted(list(set(text)))     # 등장한 글자를 중복 없이 모아서 정렬
vocab_size = len(chars)

print("Vocab:", chars)
print("vocab_size:", vocab_size)

`set(text)`는 중복을 없애고, `sorted`는 순서를 고정합니다.
순서를 고정해야 **실행할 때마다 같은 글자가 같은 번호**를 받아요.

이제 글자 ↔ 번호 사전 두 개를 만듭니다.

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}   # string to integer
itos = {i: ch for i, ch in enumerate(chars)}   # integer to string

print("stoi['안'] =", stoi["안"])
print("itos[0]    =", repr(itos[0]))
print()
for ch in "안녕":
    print(f"{ch!r} → {stoi[ch]}")

In [ ]:
def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

ids = encode("오늘 날씨")
print("encode:", ids)
print("decode:", decode(ids))
print("원래대로 돌아오나?", decode(encode(text)) == text)

**한계**: Vocab에 없는 글자가 들어오면 에러가 납니다.

In [ ]:
try:
    encode("학교")
except KeyError as e:
    print("KeyError:", e, "← 학습 텍스트에 없던 글자")

글자 단위는 Vocab이 작아서 다루기 쉽지만, **문장이 아주 길어집니다** (글자 수 = Token 수).

## 3. 단어 단위(Word) Tokenizer

반대로 **띄어쓰기 기준 단어 하나 = Token 하나**로 해보면?

In [ ]:
corpus = "나는 학교에 간다. 친구가 학교를 좋아한다. 학교가 크다."
words = corpus.split()
word_vocab = sorted(set(words))

print("단어들:", words)
print("Vocab :", word_vocab)
print("vocab_size:", len(word_vocab), "/ 문장 길이:", len(words), "Token")

문장은 짧아졌지만 문제가 보이나요?

1. **한국어는 조사가 붙어서** `학교에`, `학교를`, `학교가`가 전부 **다른 Token**이 됩니다.
   모델 입장에선 셋이 같은 "학교"라는 걸 알 길이 없어요.
2. **Vocab이 엄청 커집니다.** 실제 언어의 단어 + 활용형은 수십~수백만 개.
   Day 3에서 배웠듯 `lm_head`의 파라미터는 `n_embd × vocab_size`라서 모델이 거대해져요.
3. **처음 보는 단어(OOV, Out-Of-Vocabulary)** 는 처리할 수 없습니다.

In [ ]:
word_stoi = {w: i for i, w in enumerate(word_vocab)}
for w in ["학교에", "학교로"]:
    print(f"{w!r} →", word_stoi.get(w, "❌ 모르는 단어 (OOV)"))

## 4. 비교: 글자 vs 단어 vs Subword

| 방식 | Vocab 크기 | 문장 길이 (Token 수) | 처음 보는 단어 | 예 |
|---|---|---|---|---|
| 글자 | **작음** (수십~수천) | **김** | 글자만 알면 OK | `학`,`교`,`에` |
| 단어 | **매우 큼** (수십만+) | **짧음** | ❌ 처리 불가 | `학교에` |
| **Subword** | 적당 (3만~20만) | 적당 | 조각으로 쪼개서 OK | `학교`,`에` |

실제 LLM은 거의 전부 **Subword** 방식을 씁니다.
자주 나오는 덩어리(`학교`, `ing`, `the`)는 통째로 Token 하나, 드문 단어는 작은 조각으로 쪼개요.

> 💡 Transformer는 문장 길이 T가 길수록 계산량이 커집니다. (Day 2의 `Q @ K.T`가 `(T, T)` 표였죠!)
> 그래서 Token 수를 줄이는 게 중요하고, 그렇다고 Vocab을 무한정 키울 수도 없어서 Subword가 절충안이 됩니다.

## 5. BPE (Byte Pair Encoding) 직접 돌려보기

GPT가 쓰는 Subword 방식이 **BPE**입니다. 아이디어는 아주 단순해요.

```text
1. 처음엔 글자 단위로 시작
2. 바로 옆에 붙어있는 두 Token 쌍 중 가장 자주 나오는 쌍을 찾는다
3. 그 쌍을 새 Token 하나로 합친다 (Vocab에 추가)
4. 원하는 Vocab 크기가 될 때까지 2~3 반복
```

작은 예제로 직접 해봅시다.

In [ ]:
corpus = "학교에 학교를 학교가 학생이 학생을"
tokens = list(corpus.replace(" ", "_"))   # 공백은 보기 쉽게 _ 로

print("처음 Token:", tokens)
print("Token 수  :", len(tokens))
print("Vocab     :", sorted(set(tokens)))

### 5-1. 옆에 붙은 쌍 세기

In [ ]:
def count_pairs(tokens):
    return Counter(zip(tokens, tokens[1:]))

pairs = count_pairs(tokens)
for pair, n in pairs.most_common(5):
    print(pair, n, "번")

### 5-2. 가장 많은 쌍을 하나로 합치기

In [ ]:
def merge(tokens, pair):
    new_token = pair[0] + pair[1]
    out, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            out.append(new_token)
            i += 2                    # 두 개를 하나로 합쳤으니 두 칸 이동
        else:
            out.append(tokens[i])
            i += 1
    return out

best = pairs.most_common(1)[0][0]
tokens = merge(tokens, best)
print("합친 쌍:", best)
print("결과   :", tokens)
print("Token 수:", len(tokens))

### 5-3. 몇 번 더 반복하기

In [ ]:
tokens = list(corpus.replace(" ", "_"))
vocab = set(tokens)
print(f"시작     | Token 수 {len(tokens):2d} | Vocab {len(vocab):2d}개")

for step in range(1, 6):
    best = count_pairs(tokens).most_common(1)[0][0]
    tokens = merge(tokens, best)
    vocab.add(best[0] + best[1])
    print(f"merge {step}  | Token 수 {len(tokens):2d} | Vocab {len(vocab):2d}개 | 새 Token {best[0] + best[1]!r}")

print()
print("최종:", tokens)

**관찰**
- merge 할 때마다 **Vocab은 1개씩 늘고, 문장(Token 수)은 짧아집니다.** 4절 표의 절충이 바로 이거예요.
- 자주 나오는 `학교`, `학생`이 (앞의 공백 `_`까지 붙어서) 자연스럽게 Token 하나가 됐습니다. 아무도 "학교는 한 단어야"라고 알려주지 않았는데요!
- 실제 GPT-2는 이걸 거대한 텍스트에서 **약 5만 번** 반복해서 Vocab을 만들었습니다.

## 6. 실제 GPT Tokenizer 써보기 (`tiktoken`)

OpenAI가 공개한 `tiktoken`으로 실제 GPT Tokenizer를 써봅시다.
- `gpt2`: GPT-2의 Tokenizer (2019, Vocab 약 5만)
- `o200k_base`: 최신 GPT 모델들의 Tokenizer (Vocab 약 20만)

> 처음 실행할 때 Tokenizer 파일을 인터넷에서 내려받습니다. 실패하면 인터넷 연결을 확인하세요.

In [ ]:
try:
    import tiktoken
    gpt2 = tiktoken.get_encoding("gpt2")
    modern = tiktoken.get_encoding("o200k_base")
    print("gpt2       vocab_size:", gpt2.n_vocab)
    print("o200k_base vocab_size:", modern.n_vocab)
except Exception as e:
    gpt2 = modern = None
    print("tiktoken을 불러오지 못했어요 (인터넷 연결 확인):", e)

### 6-1. 영어 vs 한국어 Token 수 비교

In [ ]:
samples = [
    "Hello, how are you?",
    "안녕하세요, 오늘 날씨가 좋네요.",
]

for s in samples:
    print(f"{s!r}  (글자 {len(s)}개)")
    for name, enc in [("gpt2", gpt2), ("o200k_base", modern)]:
        ids = enc.encode(s)
        print(f"   {name:10s} → {len(ids):2d} Token  {ids}")
    print()

같은 길이의 문장인데 **GPT-2에서 한국어는 영어보다 몇 배나 많은 Token**을 씁니다. 최신 Tokenizer에서는 많이 나아졌어요.

### 6-2. GPT-2는 한국어를 어떻게 자르길래?
Token 하나하나를 따로 decode 해봅시다.

In [ ]:
for name, enc in [("gpt2", gpt2), ("o200k_base", modern)]:
    ids = enc.encode("안녕하세요")
    pieces = [enc.decode([i]) for i in ids]
    print(f"{name:10s} {len(ids):2d} Token: {pieces}")

GPT-2에서는 `�` (깨진 글자)가 잔뜩 나옵니다. 이유는:

1. 컴퓨터는 글자를 **바이트(byte)** 로 저장합니다. 영어 글자는 1바이트, **한글 한 글자는 3바이트**(UTF-8)예요.
2. GPT-2의 BPE는 글자가 아니라 **바이트에서 시작**합니다. (그래서 처음 보는 글자도 절대 OOV가 안 나요. 바이트는 256개뿐이니까!)
3. GPT-2 학습 데이터는 대부분 영어라서 **한글 바이트 쌍은 거의 merge되지 않았습니다.**
   → 한글 한 글자가 바이트 조각 2~3개로 쪼개지고, 조각 하나만 decode 하면 온전한 글자가 아니라 `�`가 나와요.

In [ ]:
print("'a' 바이트:", list("a".encode("utf-8")))
print("'안' 바이트:", list("안".encode("utf-8")), "← 3바이트")

> 💡 **왜 중요할까?**
> - API 요금은 Token 수로 매겨집니다 → 같은 내용이면 한국어가 더 비쌀 수 있어요.
> - 모델이 한 번에 볼 수 있는 길이(context length)도 Token 수 기준 → 한국어는 더 적은 내용만 들어가요.
> - 최신 Tokenizer가 Vocab을 20만으로 키운 이유 중 하나가 한국어 같은 비영어권 언어를 더 효율적으로 담기 위해서예요.

### 6-3. Special Token

텍스트 조각이 아니라 **특별한 의미**를 가진 Token도 있습니다.
GPT-2의 `<|endoftext|>`는 "글 하나가 여기서 끝났다"는 표시예요. 학습할 때 여러 문서를 이어붙이면서 사이에 끼워 넣습니다.

In [ ]:
eot = gpt2.eot_token
print("<|endoftext|> ID:", eot, "(GPT-2 Vocab의 맨 마지막 번호)")
ids = gpt2.encode("Hello<|endoftext|>World", allowed_special={"<|endoftext|>"})
print("encode:", ids)
print("decode:", [gpt2.decode([i]) for i in ids])

## 7. LLM과의 연결

```text
"오늘 날씨가"
   ↓  Tokenizer.encode           ← 오늘 배운 것
[1234, 5678, 910]                  Token ID (정수)
   ↓  Embedding                    ← Day 9: ID → 벡터
(T, n_embd) 벡터들
   ↓  Transformer Block × N
   ↓  lm_head (n_embd → vocab_size) ← Day 3
   ↓  Softmax                      ← Day 5: vocab_size개 Token의 확률
다음 Token ID: 1112
   ↓  Tokenizer.decode           ← 오늘 배운 것
"좋다"
```

- Tokenizer는 **모델 바깥**에 있는 전처리 도구입니다. 학습되는 파라미터가 없어요.
- 모델은 Token ID(정수)만 보고, 그 ID가 어떤 글자인지는 모릅니다.
- `vocab_size`가 모델 맨 앞(Embedding, Day 9)과 맨 끝(`lm_head`)의 크기를 결정합니다.

## 8. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** 아래 텍스트로 글자 단위 Vocab(`chars`)과 `stoi`를 만들고, `"abba"`를 encode 하세요.

In [ ]:
text = "banana bread"

chars = None     # TODO: 중복 없이 정렬된 글자 리스트
stoi = None      # TODO: 글자 → 번호 사전
encoded = None   # TODO: "abba"를 번호 리스트로

assert chars == sorted(set(text)), "chars를 다시 만들어보세요"
assert stoi == {c: i for i, c in enumerate(sorted(set(text)))}, "stoi를 다시 만들어보세요"
assert encoded == [stoi[c] for c in "abba"], "encode 결과를 다시 확인해보세요"
print("Q1 정답!", encoded)

**Q2.** `itos`를 만들고, 번호 리스트를 문자열로 되돌리는 `my_decode`를 구현하세요.

In [ ]:
text = "hello world"
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}

itos = None  # TODO

def my_decode(ids):
    return None  # TODO

ids = [stoi[c] for c in "low world"]
assert my_decode(ids) == "low world", "decode 결과가 원래 문자열과 달라요"
assert my_decode([stoi[c] for c in text]) == text
print("Q2 정답!")

**Q3.** 아래 Token 리스트에서 BPE가 **가장 먼저 합칠 쌍**은 무엇일까요? 직접 세어보고 적으세요.
(`count_pairs`를 쓰지 말고 손으로!)

In [ ]:
tokens = list("ababcab")

my_pair = "?"   # TODO: 튜플로. 예) ("x", "y")

assert my_pair == count_pairs(tokens).most_common(1)[0][0], "옆에 붙은 쌍을 하나씩 다시 세어보세요"
print("Q3 정답!", my_pair, "→", merge(tokens, my_pair))

**Q4.** GPT-2 Tokenizer로 encode 했을 때 **Token을 더 많이 쓰는 쪽**은? 실행하기 **전에** 예측하세요.
`"A"` 또는 `"B"`로 답하세요.

In [ ]:
A = "The weather is nice today."
B = "오늘 날씨가 좋다."

my_answer = "?"   # TODO: "A" 또는 "B"

n_A, n_B = len(gpt2.encode(A)), len(gpt2.encode(B))
assert my_answer == ("A" if n_A > n_B else "B"), "6-1을 다시 떠올려보세요"
print(f"Q4 정답! A: {n_A} Token (글자 {len(A)}개),  B: {n_B} Token (글자 {len(B)}개)")

## 9. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Token, Vocabulary, Token ID는 각각 무엇인가?
2. Tokenizer는 왜 필요한가? 모델 안에 있는가, 밖에 있는가?
3. 글자 단위와 단어 단위 Tokenizer의 장단점은? 실제 LLM은 왜 Subword를 쓰는가?
4. BPE는 어떤 규칙으로 Vocab을 만드는가?
5. 같은 내용인데 GPT-2에서 한국어가 영어보다 Token을 많이 쓰는 이유는?

**내 답:**

1.
2.
3.
4.
5.